# Query an Atlas with SQL

scAtlasPy provides high-level APIs for common tasks such as importing data,
preprocessing, dimensionality reduction, clustering, annotation, plotting, and
export. SQL access is a complementary interface for flexible, specialized, or
more complex questions that are not covered by those common APIs.

This tutorial shows how to use SQL to inspect, summarize, and retrieve selected
data from an Atlas without exporting the complete database. These patterns are
most useful when you need custom metadata summaries, joins across stored
analysis results, or targeted expression statistics.

`Atlas.query()` executes a DuckDB SQL statement and returns the result as a
pandas `DataFrame`. Treat it as an escape hatch for well-scoped custom queries:
filter, aggregate, and reduce data inside the database, then bring back only
the compact result needed for inspection, plotting, export, or downstream
analysis.

By the end of this tutorial, you will be able to:

- decide when SQL is useful as a complement to scAtlasPy's high-level APIs;
- inspect the tables and columns available in an Atlas;
- summarize cell metadata and quality-control results;
- retrieve expression values for selected genes;
- account for implicit zeros in sparse expression data;
- join embeddings and analysis results with cell metadata;
- export compact query results for downstream use.

## Before You Begin

This tutorial assumes that an existing Atlas has already been opened:



In [ ]:
import os
from pathlib import Path
import scatlaspy as sap

os.chdir(Path("~/scAtlaspy-code-analysis").expanduser())

atlas_path = Path("./tmp/tutorials/basic_pbmc3k/pbmc3k_basic_copy.sasql")

if not atlas_path.is_file():
    raise FileNotFoundError(f"Atlas database not found: {atlas_path}")

atlas = sap.Atlas(
    atlas_path,
    db_memory_limit="8GB",
)



The exact tables and columns available depend on which analysis steps have
already been completed.

```{important}
`Atlas.query()` returns a pandas `DataFrame`, so the rows returned by the query
are materialized in Python memory.

Use SQL to filter, aggregate, and reduce the data inside the database before
returning the result. Avoid queries such as `SELECT * FROM X_HyS_data` on a
large Atlas.
```

## 1. Inspect Available Tables

List the tables stored in the Atlas:



In [ ]:
tables = atlas.query("SHOW TABLES")
tables



Common tables include:

| Table | Typical content |
|---|---|
| `obs` | Cell metadata, quality-control metrics, filters, cluster labels, and annotations |
| `var` | Gene metadata, filters, highly variable gene flags, and gene-level statistics |
| `X_HyS_data` | Sparse count expression records imported into the Atlas |
| `X_HyS_data_data_log1p` | Derived log-normalized expression records, when created |
| `X_HyS_data_data_scale` | Derived scaled expression records, when created |
| `obsm_X_pca` | Cell-level PCA coordinates |
| `obsm_X_umap` | Cell-level UMAP coordinates |
| `varm_PCs` | Gene loadings from PCA |
| `uns_pca_stats` | PCA variance and related summary statistics |
| `rank_genes_groups` | Stored marker-ranking results |

Additional tables may be created by preprocessing, dimensionality-reduction,
clustering, annotation, or custom analysis functions.

Some tables, such as `X_HyS_indptr` and related filtered index tables, support
the internal sparse data representation. They normally do not need to be
queried directly for routine analysis.

## 2. Inspect Table Schemas

Before writing a query, inspect the columns available in the relevant tables:



In [ ]:
atlas.query("PRAGMA table_info(obs)")


In [ ]:
atlas.query("PRAGMA table_info(var)")



For expression queries, inspect the base count table and any derived expression table you plan to query:



In [ ]:
atlas.query("PRAGMA table_info(X_HyS_data)")


In [ ]:
atlas.query("PRAGMA table_info(X_HyS_data_data_log1p)")



This is particularly important for expression representations. `X_HyS_data` stores imported count-scale values in `data_count`. Derived representations are materialized in separate tables named `X_HyS_data_<field>`, for example `X_HyS_data_data_log1p` for `data_log1p` and `X_HyS_data_data_scale` for `data_scale`.

Use the expression table and value column appropriate for the analysis.

```{note}
Column names such as `scatlas_cluster`, `cell_total_counts`, or `pct_counts_mt` appear
only after the corresponding workflow steps have created them. Adapt the
queries below to the columns present in your Atlas.
```

## 3. Summarize Cell and Gene Metadata

Count the cells and genes stored in the Atlas:



In [ ]:
atlas_size = atlas.query("""
    SELECT
        (SELECT COUNT(*) FROM obs) AS n_cells,
        (SELECT COUNT(*) FROM var) AS n_genes
""")

atlas_size



Summarize the number of cells in each cluster:



In [ ]:
cluster_size = atlas.query("""
    SELECT
        scatlas_cluster,
        COUNT(*) AS n_cells
    FROM obs
    WHERE scatlas_cluster IS NOT NULL
    GROUP BY scatlas_cluster
    ORDER BY scatlas_cluster
""")

cluster_size



To examine how manual annotations are distributed across clusters:



In [ ]:
cluster_by_annotation = atlas.query("""
    SELECT
        cell_type_manual,
        scatlas_cluster,
        COUNT(*) AS n_cells
    FROM obs
    WHERE cell_type_manual IS NOT NULL
      AND scatlas_cluster IS NOT NULL
    GROUP BY cell_type_manual, scatlas_cluster
    ORDER BY cell_type_manual, scatlas_cluster
""")

cluster_by_annotation



Summarize quality-control metrics by cluster:



In [ ]:
cluster_qc = atlas.query("""
    SELECT
        scatlas_cluster,
        COUNT(*) AS n_cells,
        MEDIAN(cell_total_counts) AS median_counts,
        MEDIAN(n_genes_by_counts) AS median_genes,
        MEDIAN(pct_counts_mt) AS median_pct_mt
    FROM obs
    WHERE scatlas_cluster IS NOT NULL
    GROUP BY scatlas_cluster
    ORDER BY scatlas_cluster
""")

cluster_qc



These summaries are calculated inside the database. Only the compact aggregated
table is returned to Python.

## 4. Retrieve Expression for Selected Genes

Expression values are stored sparsely in `X_HyS_data`. Join the expression
records with `var` to identify genes and with `obs` to retrieve cell metadata.

For example, retrieve nonzero log-transformed expression records for three
marker genes:



In [ ]:
gene_expr = atlas.query("""
    SELECT
        x.atlas_cell_id,
        o.scatlas_cluster,
        v.atlas_gene_name,
        x.data_log1p
    FROM X_HyS_data_data_log1p AS x
    JOIN var AS v
      ON x.atlas_gene_id = v.atlas_gene_id
    JOIN obs AS o
      ON x.atlas_cell_id = o.atlas_cell_id
    WHERE v.atlas_gene_name IN ('MS4A1', 'LYZ', 'NKG7')
    ORDER BY x.atlas_cell_id, v.atlas_gene_name
    LIMIT 100
""")

gene_expr



```{important}
This query returns only expression records present in the sparse table. Cells
with implicit zero expression for a selected gene do not appear in the result.
```

Use persistent numeric identifiers such as `atlas_cell_id` and
`atlas_gene_id` for joins. Gene names are more suitable for selecting and
displaying genes than for joining large tables.

## 5. Calculate Mean Nonzero Expression

To calculate the average expression among cells with a stored nonzero record:



In [ ]:
mean_nonzero = atlas.query("""
    SELECT
        o.scatlas_cluster,
        v.atlas_gene_name,
        AVG(x.data_log1p) AS mean_nonzero_log1p,
        COUNT(*) AS n_expressing
    FROM X_HyS_data_data_log1p AS x
    JOIN var AS v
      ON x.atlas_gene_id = v.atlas_gene_id
    JOIN obs AS o
      ON x.atlas_cell_id = o.atlas_cell_id
    WHERE v.atlas_gene_name IN ('MS4A1', 'LYZ', 'NKG7')
      AND o.scatlas_cluster IS NOT NULL
    GROUP BY o.scatlas_cluster, v.atlas_gene_name
    ORDER BY v.atlas_gene_name, o.scatlas_cluster
""")

mean_nonzero



This result answers:

> Among cells with nonzero expression of the gene, what is the mean
> `data_log1p` value?

It does not calculate the mean across every cell in the cluster.

## 6. Include Implicit Zeros

Sparse expression tables omit zero-valued records. To calculate the percentage
of expressing cells or the mean across all cells, the denominator must include
cells with implicit zeros.

The following query first aggregates the nonzero records and then combines them
with every cluster and selected gene:



In [ ]:
expression_summary = atlas.query("""
    WITH selected_genes AS (
        SELECT
            atlas_gene_id,
            atlas_gene_name
        FROM var
        WHERE atlas_gene_name IN ('MS4A1', 'LYZ', 'NKG7')
    ),

    cluster_sizes AS (
        SELECT
            scatlas_cluster,
            COUNT(*) AS n_cells
        FROM obs
        WHERE scatlas_cluster IS NOT NULL
        GROUP BY scatlas_cluster
    ),

    nonzero_summary AS (
        SELECT
            o.scatlas_cluster,
            x.atlas_gene_id,
            COUNT(*) AS n_expressing,
            SUM(x.data_log1p) AS sum_log1p
        FROM X_HyS_data_data_log1p AS x
        JOIN selected_genes AS g
          ON x.atlas_gene_id = g.atlas_gene_id
        JOIN obs AS o
          ON x.atlas_cell_id = o.atlas_cell_id
        WHERE o.scatlas_cluster IS NOT NULL
        GROUP BY o.scatlas_cluster, x.atlas_gene_id
    )

    SELECT
        c.scatlas_cluster,
        g.atlas_gene_name,
        c.n_cells,
        COALESCE(n.n_expressing, 0) AS n_expressing,
        100.0 * COALESCE(n.n_expressing, 0) / c.n_cells
            AS pct_expressing,
        COALESCE(n.sum_log1p, 0.0) / c.n_cells
            AS mean_log1p_all_cells
    FROM cluster_sizes AS c
    CROSS JOIN selected_genes AS g
    LEFT JOIN nonzero_summary AS n
      ON n.scatlas_cluster = c.scatlas_cluster
     AND n.atlas_gene_id = g.atlas_gene_id
    ORDER BY g.atlas_gene_name, c.scatlas_cluster
""")

expression_summary



This query reports:

- the number of cells in each cluster;
- the number and percentage of cells with a nonzero expression record;
- the mean log-transformed expression across all cells, treating missing sparse
  records as zero.

The query creates combinations only between clusters and the selected genes. It
does not construct a complete cell-by-gene table.

```{note}
The interpretation of an implicit missing record depends on the expression
representation. For count and `log1p` fields, an omitted sparse record generally
represents zero expression.

Do not automatically apply the same interpretation to a transformed
representation whose zero values or sparsity semantics differ.
```

Also note that the mean of `log1p` values is not the same as applying `log1p`
to the mean count. Choose the statistic that matches the biological or
visualization question.

## 7. Restrict Queries to Selected Cells or Genes

Metadata conditions can be added directly to expression queries.

For example, restrict the summary to cells passing quality-control filtering:

```sql
AND o.filter_cells = TRUE
```

You can also restrict genes using metadata stored in `var`:

```sql
AND v.filter_genes = TRUE
AND v.highly_variable_genes = TRUE
```

Apply the same population definition consistently when comparing SQL summaries
with PCA, clustering, visualization, or streaming computations.

## 8. Query Embeddings

Join stored UMAP coordinates with cell metadata:



In [ ]:
umap_preview = atlas.query("""
    SELECT
        u.atlas_cell_id,
        u.umap1,
        u.umap2,
        o.scatlas_cluster,
        o.cell_total_counts,
        o.n_genes_by_counts
    FROM obsm_X_umap AS u
    JOIN obs AS o
      ON u.atlas_cell_id = o.atlas_cell_id
    ORDER BY u.atlas_cell_id
    LIMIT 10000
""")

umap_preview.head()



The `LIMIT` clause makes this a deterministic preview of the first 10,000
stored cells. For representative visualization of a large Atlas, use the
sampling options provided by scAtlasPy plotting functions rather than relying
on the first rows of a table.

Inspect the embedding schema first if the coordinate column names differ:



In [ ]:
atlas.query("PRAGMA table_info(obsm_X_umap)")



## 9. Inspect Stored Analysis Results

Result-table schemas may depend on the analysis function and its parameters.

For example, inspect the marker-ranking table before writing a query against it:



In [ ]:
atlas.query("PRAGMA table_info(rank_genes_groups)")



Preview a small number of records:



In [ ]:
marker_preview = atlas.query("""
    SELECT *
    FROM rank_genes_groups
    LIMIT 20
""")

marker_preview



After identifying the relevant columns, use SQL to select clusters, rank genes,
filter by score, or join marker results with gene metadata.

## 10. Export Query Results

Because `Atlas.query()` returns a pandas `DataFrame`, query results can be
written directly to common tabular formats:



In [ ]:
from pathlib import Path

output_dir = Path("./results")
output_dir.mkdir(parents=True, exist_ok=True)

cluster_qc.to_csv(
    output_dir / "cluster_qc.csv",
    index=False,
)

expression_summary.to_csv(
    output_dir / "marker_expression_by_cluster.csv",
    index=False,
)



The returned DataFrame can also be passed to pandas, Matplotlib, or another
downstream analysis library.

## Write Scalable SQL Queries

When querying a large Atlas:

- select only the columns needed for the analysis;
- filter cells and genes before joining large tables;
- aggregate inside SQL before returning results to Python;
- use `LIMIT` while inspecting unfamiliar tables;
- join expression data by numeric cell and gene identifiers;
- avoid returning complete sparse expression tables as DataFrames;
- avoid constructing complete cell-by-gene combinations unless the selected
  dimensions are small.

SQL is particularly useful for metadata summaries, stored analysis results, and
expression statistics involving a limited set of genes.

For algorithms that repeatedly traverse many genes across large numbers of
cells, use the Atlas streaming interfaces instead of materializing the result
of a large SQL query.



## Close the Atlas

Close the database connection when this tutorial is complete. This releases
the DuckDB file lock so the same `.sasql` Atlas can be opened by another
notebook or Python session.


In [ ]:
atlas.close()


## Next Steps

Continue with {doc}`stream-mean-and-variance` to calculate statistics by
traversing atlas-scale expression data in minibatches.

See {doc}`visualize-analysis-results` to inspect stored metadata, embeddings,
cluster assignments, and marker results with scAtlasPy plotting functions.

For SQL syntax, expressions, joins, aggregations, and other DuckDB-specific
features, see the [DuckDB SQL documentation](https://duckdb.org/docs/current/sql/introduction).